# Módulo 03 · Aula 04 — Manutenção e Transações

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

Até aqui você só **leu** do banco. Consultas erradas geram relatórios errados — ruim, mas reversível.

A partir de agora você vai **escrever**. E escrita errada destrói dados.

Esta aula é sobre fazer isso com segurança.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | `INSERT` | Inserir uma linha, muitas, ou o resultado de uma consulta |
| 2 | **Consultas parametrizadas** | 🔴 SQL injection — a vulnerabilidade nº 1 |
| 3 | `UPDATE` | E o desastre do `WHERE` esquecido |
| 4 | `DELETE` e *soft delete* | Apagar de verdade nem sempre é o certo |
| 5 | `UPSERT` (`ON CONFLICT`) | "Insere ou atualiza" numa operação só |
| 6 | Índices | Consultas 1.000× mais rápidas — com um custo |
| 7 | `EXPLAIN QUERY PLAN` | Ver como o banco decidiu executar |
| 8 | **Transações e ACID** | Ou tudo acontece, ou nada acontece |
| 9 | `sqlite3` no Python | Cuidados de produção |

## ⚙️ Preparando o banco

Mesma base das aulas anteriores.

> ▶️ **Execute esta célula primeiro.**

In [ ]:
import sqlite3
import random

# ═══════════════════════════════════════════════════════════════
#  Banco de treino da Aurora Comércio
#  Execute esta célula UMA VEZ, antes de qualquer outra.
#  Ela é idempotente: pode rodar de novo a qualquer momento.
# ═══════════════════════════════════════════════════════════════

CONN = sqlite3.connect(":memory:")
CONN.execute("PRAGMA foreign_keys = ON")

CONN.executescript("""
CREATE TABLE categorias (
    id           INTEGER PRIMARY KEY,
    nome         TEXT    NOT NULL UNIQUE,
    margem_alvo  REAL    NOT NULL DEFAULT 0.25 CHECK (margem_alvo BETWEEN 0 AND 1)
);

CREATE TABLE produtos (
    id           INTEGER PRIMARY KEY,
    sku          TEXT    NOT NULL UNIQUE,
    nome         TEXT    NOT NULL,
    categoria_id INTEGER NOT NULL REFERENCES categorias(id) ON DELETE RESTRICT,
    preco        REAL    NOT NULL CHECK (preco >= 0),
    custo        REAL    NOT NULL CHECK (custo >= 0),
    estoque      INTEGER NOT NULL DEFAULT 0 CHECK (estoque >= 0),
    ativo        INTEGER NOT NULL DEFAULT 1 CHECK (ativo IN (0,1))
);

CREATE TABLE clientes (
    id            INTEGER PRIMARY KEY,
    nome          TEXT NOT NULL,
    email         TEXT NOT NULL UNIQUE,
    cidade        TEXT NOT NULL,
    uf            TEXT NOT NULL CHECK (length(uf) = 2),
    segmento      TEXT NOT NULL DEFAULT 'varejo'
                       CHECK (segmento IN ('varejo','corporativo')),
    data_cadastro TEXT NOT NULL,
    telefone      TEXT
);

CREATE TABLE pedidos (
    id          INTEGER PRIMARY KEY,
    cliente_id  INTEGER NOT NULL REFERENCES clientes(id) ON DELETE RESTRICT,
    data_pedido TEXT    NOT NULL,
    status      TEXT    NOT NULL CHECK (status IN ('pago','pendente','cancelado')),
    canal       TEXT    NOT NULL CHECK (canal IN ('site','app','marketplace')),
    frete       REAL    NOT NULL DEFAULT 0 CHECK (frete >= 0)
);

CREATE TABLE itens_pedido (
    id             INTEGER PRIMARY KEY,
    pedido_id      INTEGER NOT NULL REFERENCES pedidos(id)  ON DELETE CASCADE,
    produto_id     INTEGER NOT NULL REFERENCES produtos(id) ON DELETE RESTRICT,
    quantidade     INTEGER NOT NULL CHECK (quantidade > 0),
    preco_unitario REAL    NOT NULL CHECK (preco_unitario >= 0),
    UNIQUE (pedido_id, produto_id)
);
""")

_CATEGORIAS = [(1, "Notebooks", 0.18), (2, "Monitores", 0.22), (3, "Periféricos", 0.38),
               (4, "Armazenamento", 0.30), (5, "Redes", 0.28), (6, "Áudio", 0.35)]

_PRODUTOS = [
    ("NB-DELL-15",  "Notebook Dell Inspiron 15",     1, 2599.90, 2120.00,  14),
    ("NB-ACER-N5",  "Notebook Acer Nitro 5",         1, 3299.00, 2780.00,   7),
    ("NB-LEN-IP3",  "Notebook Lenovo IdeaPad 3",     1, 2199.00, 1850.00,  22),
    ("NB-APPL-M2",  "MacBook Air M2",                1, 9499.00, 8300.00,   3),
    ("MO-LG-24UW",  "Monitor LG 24 UltraWide",       2, 1199.00,  920.00,  31),
    ("MO-SAM-ODY",  "Monitor Samsung Odyssey 27",    2, 1849.00, 1420.00,  12),
    ("MO-AOC-22",   "Monitor AOC 22 Full HD",        2,  749.00,  560.00,  45),
    ("PE-LOG-MX3",  "Mouse Logitech MX Master 3",    3,  549.00,  340.00,  88),
    ("PE-LOG-M170", "Mouse Logitech M170",           3,   89.90,   52.00, 240),
    ("PE-RED-K552", "Teclado Redragon K552",         3,  249.00,  150.00,  64),
    ("PE-LOG-C920", "Webcam Logitech C920",          3,  449.00,  290.00,  37),
    ("AU-HYP-CL2",  "Headset HyperX Cloud II",       6,  399.00,  255.00,  29),
    ("AU-JBL-T510", "Fone JBL Tune 510BT",           6,  229.00,  140.00,  73),
    ("AR-SSD-1TB",  "SSD NVMe 1TB Kingston",         4,  489.00,  360.00,  52),
    ("AR-SSD-480",  "SSD SATA 480GB Sandisk",        4,  229.00,  158.00,  96),
    ("AR-HD-2TB",   "HD Externo 2TB Seagate",        4,  549.00,  410.00,  18),
    ("AR-PEN-128",  "Pendrive 128GB Sandisk",        4,   79.90,   44.00, 180),
    ("RE-TPL-AX55", "Roteador TP-Link Archer AX55",  5,  699.00,  505.00,  26),
    ("RE-TPL-RE30", "Repetidor TP-Link RE305",       5,  229.00,  152.00,  41),
    ("RE-INT-AX20", "Placa de Rede Intel AX200",     5,  189.00,  124.00,  33),
]

_NOMES = ["Ana Costa", "Bruno Rocha", "Carla Dias", "Daniel Souza", "Elisa Martins",
          "Fábio Nunes", "Gustavo Reis", "Helena Prado", "Igor Batista", "Julia Andrade",
          "Lucas Moreira", "Maria Souza", "Nathalia Freitas", "Otávio Pinto",
          "Priscila Gomes", "Rafael Torres", "Sabrina Melo", "Thiago Barros",
          "Vanessa Lima", "William Cruz", "Beatriz Almeida", "Caio Ferreira",
          "Débora Ramos", "Eduardo Pires", "Fernanda Vieira", "Gabriel Mendes",
          "Isabela Rocha", "João Lima", "Karina Duarte", "Leonardo Castro"]

_CIDADES = [("Campinas", "SP"), ("São Paulo", "SP"), ("Sorocaba", "SP"),
            ("Ribeirão Preto", "SP"), ("Jundiaí", "SP"), ("Santos", "SP"),
            ("Belo Horizonte", "MG"), ("Uberlândia", "MG"), ("Curitiba", "PR"),
            ("Londrina", "PR"), ("Porto Alegre", "RS"), ("Florianópolis", "SC"),
            ("Rio de Janeiro", "RJ"), ("Niterói", "RJ"), ("Salvador", "BA"),
            ("Recife", "PE"), ("Fortaleza", "CE"), ("Brasília", "DF"),
            ("Goiânia", "GO"), ("Vitória", "ES")]

_rnd = random.Random(42)     # semente fixa = todos veem os mesmos números

CONN.executemany("INSERT INTO categorias VALUES (?,?,?)", _CATEGORIAS)
CONN.executemany(
    "INSERT INTO produtos (sku,nome,categoria_id,preco,custo,estoque,ativo) "
    "VALUES (?,?,?,?,?,?,1)", _PRODUTOS)

_acentos = str.maketrans("áéíóúãõâêôç", "aeiouaoaeoc")
_clientes = []
for _i, _nome in enumerate(_NOMES, 1):
    _cidade, _uf = _rnd.choice(_CIDADES)
    _login = _nome.split()[0].lower().translate(_acentos)
    _clientes.append((
        _i, _nome, f"{_login}{_i}@email.com", _cidade, _uf,
        "corporativo" if _rnd.random() < 0.25 else "varejo",
        f"2026-{_rnd.randint(1, 6):02d}-{_rnd.randint(1, 28):02d}",
        f"(19) 9{_rnd.randint(1000, 9999)}-{_rnd.randint(1000, 9999)}" if _rnd.random() < 0.6 else None,
    ))
CONN.executemany("INSERT INTO clientes VALUES (?,?,?,?,?,?,?,?)", _clientes)

_pedidos, _itens, _id_item = [], [], 0
# Os 3 últimos clientes ficam SEM pedido de propósito: toda base real tem
# gente que se cadastrou e nunca comprou, e você precisa saber encontrá-los.
for _pid in range(1, 181):
    _mes = _rnd.choices([5, 6, 7], weights=[2, 3, 4])[0]
    _pedidos.append((
        _pid, _rnd.randint(1, len(_NOMES) - 3),
        f"2026-{_mes:02d}-{_rnd.randint(1, 28):02d}",
        _rnd.choices(["pago", "pendente", "cancelado"], weights=[80, 12, 8])[0],
        _rnd.choices(["site", "app", "marketplace"], weights=[50, 30, 20])[0],
        _rnd.choice([0.0, 9.90, 19.90, 29.90]),
    ))
    _n_itens = _rnd.choices([1, 2, 3, 4], weights=[45, 30, 17, 8])[0]
    for _prod in _rnd.sample(range(1, len(_PRODUTOS) + 1), _n_itens):
        _id_item += 1
        _itens.append((
            _id_item, _pid, _prod,
            _rnd.choices([1, 2, 3, 5, 10], weights=[55, 22, 12, 7, 4])[0],
            round(_PRODUTOS[_prod - 1][3] * _rnd.choice([1.0, 1.0, 1.0, 0.95, 0.90]), 2),
        ))
CONN.executemany("INSERT INTO pedidos VALUES (?,?,?,?,?,?)", _pedidos)
CONN.executemany("INSERT INTO itens_pedido VALUES (?,?,?,?,?)", _itens)
CONN.commit()


# ── Funções auxiliares ───────────────────────────────────────
def _fmt(valor):
    if valor is None:
        return "NULL"
    if isinstance(valor, float):
        return f"{valor:,.2f}"
    if isinstance(valor, int):
        return f"{valor:,}"
    return str(valor)


def sql(consulta, parametros=(), limite=30):
    """Executa uma consulta e imprime o resultado formatado."""
    try:
        cursor = CONN.execute(consulta, parametros)
    except sqlite3.Error as erro:
        print(f"❌ {type(erro).__name__}: {erro}")
        return None

    if cursor.description is None:
        CONN.commit()
        print(f"✅ OK — {cursor.rowcount} linha(s) afetada(s)" if cursor.rowcount >= 0 else "✅ OK")
        return None

    colunas = [d[0] for d in cursor.description]
    linhas = cursor.fetchall()
    total = len(linhas)
    linhas = linhas[:limite]
    if not linhas:
        print("(nenhuma linha)")
        return []

    texto = [[_fmt(v) for v in linha] for linha in linhas]
    numerica = [
        any(isinstance(l[i], (int, float)) for l in linhas)
        and all(isinstance(l[i], (int, float)) or l[i] is None for l in linhas)
        for i in range(len(colunas))
    ]
    larguras = [max(len(colunas[i]), max(len(l[i]) for l in texto))
                for i in range(len(colunas))]

    def borda(e, m, d):
        return e + m.join("─" * (w + 2) for w in larguras) + d

    print(borda("┌", "┬", "┐"))
    print("│ " + " │ ".join(c.ljust(w) for c, w in zip(colunas, larguras)) + " │")
    print(borda("├", "┼", "┤"))
    for linha in texto:
        print("│ " + " │ ".join(
            (v.rjust(w) if numerica[i] else v.ljust(w))
            for i, (v, w) in enumerate(zip(linha, larguras))) + " │")
    print(borda("└", "┴", "┘"))
    print(f"{total} linha(s)" + (f" — exibindo as {limite} primeiras" if total > limite else ""))
    return linhas


def ddl(script):
    """Executa um script com vários comandos."""
    try:
        CONN.executescript(script)
        CONN.commit()
        print("✅ Script executado")
    except sqlite3.Error as erro:
        print(f"❌ {type(erro).__name__}: {erro}")


print("✅ Banco da Aurora criado em memória\n")
sql("""
SELECT 'categorias'   AS tabela, COUNT(*) AS linhas FROM categorias
UNION ALL SELECT 'produtos',     COUNT(*) FROM produtos
UNION ALL SELECT 'clientes',     COUNT(*) FROM clientes
UNION ALL SELECT 'pedidos',      COUNT(*) FROM pedidos
UNION ALL SELECT 'itens_pedido', COUNT(*) FROM itens_pedido
""")

## 1. `INSERT`

```sql
INSERT INTO tabela (col1, col2) VALUES (v1, v2);
```

**Sempre liste as colunas.** `INSERT INTO t VALUES (...)` depende da ordem física das colunas — e quebra silenciosamente no dia em que alguém adicionar uma coluna nova.

In [ ]:
sql("""
INSERT INTO categorias (nome, margem_alvo)
VALUES ('Acessórios', 0.42)
""")
sql("SELECT * FROM categorias ORDER BY id")

In [ ]:
# Inserção múltipla: uma transação só, MUITO mais rápido que N comandos
sql("""
INSERT INTO produtos (sku, nome, categoria_id, preco, custo, estoque) VALUES
    ('AC-SUP-NB1', 'Suporte para Notebook',      7, 129.00,  78.00, 50),
    ('AC-HUB-USB', 'Hub USB-C 7 portas',         7, 199.00, 121.00, 34),
    ('AC-CAB-HDMI','Cabo HDMI 2.1 2m',           7,  69.90,  38.00, 120)
""")
sql("SELECT sku, nome, preco, estoque FROM produtos WHERE categoria_id = 7")

### `INSERT ... SELECT` — inserindo o resultado de uma consulta

É como se popula tabelas derivadas, faz backup e move dados entre tabelas — tudo dentro do banco, sem trafegar nada para a aplicação.

In [ ]:
ddl("""
CREATE TABLE resumo_mensal (
    mes          TEXT    NOT NULL,
    canal        TEXT    NOT NULL,
    pedidos      INTEGER NOT NULL,
    faturamento  REAL    NOT NULL,
    PRIMARY KEY (mes, canal)
);
""")

sql("""
INSERT INTO resumo_mensal (mes, canal, pedidos, faturamento)
SELECT
    strftime('%Y-%m', p.data_pedido),
    p.canal,
    COUNT(DISTINCT p.id),
    ROUND(SUM(i.quantidade * i.preco_unitario), 2)
FROM pedidos p
JOIN itens_pedido i ON i.pedido_id = p.id
WHERE p.status = 'pago'
GROUP BY strftime('%Y-%m', p.data_pedido), p.canal
""")

sql("SELECT * FROM resumo_mensal ORDER BY mes, faturamento DESC")

### `RETURNING` — recuperando o que foi gravado

Devolve as linhas afetadas. Muito útil para pegar o `id` gerado sem uma segunda consulta.

Disponível no SQLite a partir da versão **3.35** (2021).

In [ ]:
sql("""
INSERT INTO clientes (nome, email, cidade, uf, segmento, data_cadastro)
VALUES ('Novo Cliente Teste', 'teste@email.com', 'Campinas', 'SP', 'varejo', '2026-08-12')
RETURNING id, nome, email
""")

## 2. 🔴 Consultas parametrizadas — SQL injection

Este é o assunto mais importante da aula.

Quando você monta SQL **concatenando strings**, o dado do usuário vira **código**. Um atacante pode fechar a string e escrever o que quiser.

```python
# 🔴 NUNCA FAÇA ISSO
consulta = f"SELECT * FROM clientes WHERE email = '{email}'"
```

Se `email` for `x' OR '1'='1`, a consulta vira:

```sql
SELECT * FROM clientes WHERE email = 'x' OR '1'='1'
```

...e devolve **a base inteira**.

In [ ]:
# Demonstração controlada da vulnerabilidade
entrada_maliciosa = "x' OR '1'='1"

consulta_vulneravel = f"SELECT id, nome, email FROM clientes WHERE email = '{entrada_maliciosa}'"
print("SQL gerado:")
print(" ", consulta_vulneravel)
print()
sql(consulta_vulneravel, limite=5)

In [ ]:
# ✅ A forma correta: placeholders. O valor NUNCA vira código.
entrada_maliciosa = "x' OR '1'='1"

print("Com parâmetro (o ataque falha):")
sql("SELECT id, nome, email FROM clientes WHERE email = ?", (entrada_maliciosa,))

print("\nCom um e-mail real:")
sql("SELECT id, nome, email FROM clientes WHERE email = ?", ("teste@email.com",))

### Como funciona o placeholder

O driver envia o comando e os valores **separadamente**. O banco compila a estrutura da consulta primeiro e só depois encaixa os valores — que são tratados como **dados**, jamais como sintaxe. Não há como escapar dessa separação.

**Bônus:** consultas parametrizadas são **mais rápidas** em execuções repetidas, porque o plano compilado é reaproveitado.

| Estilo | Sintaxe | Exemplo |
|--------|---------|---------|
| Posicional | `?` | `execute("... WHERE id = ?", (5,))` |
| Nomeado | `:nome` | `execute("... WHERE id = :id", {"id": 5})` |

> ⚠️ **A vírgula em `(5,)` é obrigatória.** Sem ela, `(5)` é apenas o número 5 entre parênteses, não uma tupla — e o driver reclama.

In [ ]:
# Placeholders nomeados: mais legíveis quando há muitos parâmetros
sql("""
SELECT sku, nome, preco, estoque
FROM produtos
WHERE categoria_id = :cat
  AND preco BETWEEN :min AND :max
  AND estoque > :estoque_min
ORDER BY preco
""", {"cat": 3, "min": 100, "max": 600, "estoque_min": 30})

> 🔴 **O que NÃO pode ser parametrizado:** nomes de tabela e de coluna. `SELECT * FROM ?` não funciona — a estrutura precisa ser conhecida na compilação.
>
> Se você realmente precisa de nome de coluna dinâmico (ordenação escolhida pelo usuário, por exemplo), valide contra uma **lista branca**:
> ```python
> COLUNAS_PERMITIDAS = {"nome", "preco", "estoque"}
> if coluna not in COLUNAS_PERMITIDAS:
>     raise ValueError(f"coluna inválida: {coluna}")
> consulta = f"SELECT * FROM produtos ORDER BY {coluna}"
> ```

## 3. `UPDATE`

```sql
UPDATE tabela
SET coluna = valor
WHERE condicao;
```

In [ ]:
sql("SELECT sku, nome, preco FROM produtos WHERE sku = 'AC-CAB-HDMI'")
sql("UPDATE produtos SET preco = 79.90 WHERE sku = 'AC-CAB-HDMI'")
sql("SELECT sku, nome, preco FROM produtos WHERE sku = 'AC-CAB-HDMI'")

In [ ]:
# Várias colunas, e expressões que usam o valor atual
sql("""
UPDATE produtos
SET preco   = ROUND(preco * 1.08, 2),
    estoque = estoque + 10
WHERE categoria_id = 7
""")
sql("SELECT sku, nome, preco, estoque FROM produtos WHERE categoria_id = 7")

### 🔴 O `UPDATE` sem `WHERE`

Um `UPDATE` sem `WHERE` altera **todas as linhas da tabela**. Não há confirmação, não há aviso, não há desfazer.

**O protocolo profissional, sempre:**

1. Escreva primeiro como `SELECT` com o mesmo `WHERE`.
2. Confira quantas linhas voltaram — é o número que você espera?
3. **Só então** troque `SELECT ...` por `UPDATE ... SET ...`.
4. Se possível, execute dentro de uma transação (seção 8).

In [ ]:
# PASSO 1: confira o alcance antes
sql("""
SELECT COUNT(*) AS linhas_que_serao_afetadas
FROM produtos
WHERE categoria_id = 7 AND preco > 100
""")

In [ ]:
# PASSO 2: agora sim, com o WHERE conferido
sql("""
UPDATE produtos
SET preco = ROUND(preco * 0.90, 2)
WHERE categoria_id = 7 AND preco > 100
""")
sql("SELECT sku, preco FROM produtos WHERE categoria_id = 7")

### `UPDATE` com dados de outra tabela

O SQLite não tem `UPDATE ... FROM` em versões antigas. A forma portável usa subconsulta correlacionada.

In [ ]:
# Alinhar o preço de catálogo ao maior preço realmente praticado nos últimos pedidos
sql("""
UPDATE produtos
SET preco = (
    SELECT MAX(i.preco_unitario)
    FROM itens_pedido i
    JOIN pedidos p ON p.id = i.pedido_id
    WHERE i.produto_id = produtos.id
      AND p.status = 'pago'
)
WHERE EXISTS (
    SELECT 1 FROM itens_pedido i2 WHERE i2.produto_id = produtos.id
)
  AND categoria_id = 5
""")

sql("SELECT sku, nome, preco FROM produtos WHERE categoria_id = 5")

> ⚠️ **O `WHERE EXISTS` no final é essencial.** Sem ele, produtos sem venda receberiam `NULL` da subconsulta — e o `NOT NULL` da coluna faria o comando inteiro falhar (ou, pior, num schema permissivo, gravaria `NULL` silenciosamente).

## 4. `DELETE` e *soft delete*

```sql
DELETE FROM tabela WHERE condicao;
```

As mesmas regras do `UPDATE` valem, com mais força: **`DELETE` sem `WHERE` esvazia a tabela.**

| Comando | Faz |
|---------|-----|
| `DELETE FROM t WHERE ...` | Remove linhas específicas |
| `DELETE FROM t` | Remove **todas** as linhas (a tabela continua existindo) |
| `DROP TABLE t` | Remove a tabela inteira, estrutura incluída |

In [ ]:
sql("SELECT COUNT(*) AS antes FROM clientes")
sql("DELETE FROM clientes WHERE email = 'teste@email.com'")
sql("SELECT COUNT(*) AS depois FROM clientes")

In [ ]:
# O CASCADE em ação: apagar um pedido leva os itens junto
sql("SELECT COUNT(*) AS itens_antes FROM itens_pedido")
sql("SELECT COUNT(*) AS itens_do_pedido_1 FROM itens_pedido WHERE pedido_id = 1")
sql("DELETE FROM pedidos WHERE id = 1")
sql("SELECT COUNT(*) AS itens_depois FROM itens_pedido")

### *Soft delete* — apagar sem apagar

Em sistemas reais, raramente se apaga de verdade. Motivos:

- **Auditoria e legislação.** Nota fiscal cancelada não some do sistema.
- **Integridade histórica.** Apagar um produto quebraria os pedidos antigos.
- **Erro humano.** "Apaguei sem querer" tem conserto se o dado ainda estiver lá.

A solução é uma coluna `ativo` ou `deletado_em`.

In [ ]:
ddl("ALTER TABLE produtos ADD COLUMN deletado_em TEXT")

# "Apagar" = marcar
sql("""
UPDATE produtos
SET ativo = 0,
    deletado_em = '2026-08-12'
WHERE sku = 'AC-CAB-HDMI'
""")

print("Catálogo visível (o padrão de toda consulta do sistema):")
sql("SELECT sku, nome, preco, ativo FROM produtos WHERE deletado_em IS NULL AND categoria_id = 7")

print("\nIncluindo os removidos (só para auditoria):")
sql("SELECT sku, nome, ativo, deletado_em FROM produtos WHERE categoria_id = 7")

> 💡 **O custo do soft delete:** **toda** consulta do sistema precisa lembrar de filtrar `WHERE deletado_em IS NULL`. Esquecer uma é mostrar dado apagado ao usuário.
>
> A solução profissional é criar uma **VIEW** com o filtro embutido e fazer a aplicação consultar sempre a view, nunca a tabela.

In [ ]:
ddl("""
CREATE VIEW produtos_ativos AS
SELECT id, sku, nome, categoria_id, preco, custo, estoque
FROM produtos
WHERE deletado_em IS NULL AND ativo = 1;
""")

sql("SELECT COUNT(*) AS na_tabela FROM produtos")
sql("SELECT COUNT(*) AS na_view FROM produtos_ativos")

## 5. `UPSERT` — inserir ou atualizar

*"Se já existe, atualiza; se não, insere."* Sem `UPSERT`, isso exigiria um `SELECT`, um `if` e dois caminhos — com uma janela de concorrência no meio.

```sql
INSERT INTO tabela (...) VALUES (...)
ON CONFLICT (coluna_unica) DO UPDATE SET
    coluna = excluded.coluna;
```

`excluded` é uma pseudo-tabela com os valores que **tentaram** ser inseridos.

É a operação central de qualquer **pipeline de ingestão idempotente** — que é exatamente o que você vai construir no Módulo 10.

In [ ]:
sql("SELECT * FROM resumo_mensal WHERE mes = '2026-07' ORDER BY canal")

In [ ]:
# Reprocessando o mesmo mês: em vez de apagar e reinserir, faz UPSERT
sql("""
INSERT INTO resumo_mensal (mes, canal, pedidos, faturamento)
SELECT
    strftime('%Y-%m', p.data_pedido),
    p.canal,
    COUNT(DISTINCT p.id),
    ROUND(SUM(i.quantidade * i.preco_unitario), 2)
FROM pedidos p
JOIN itens_pedido i ON i.pedido_id = p.id
WHERE p.status = 'pago'
  AND strftime('%Y-%m', p.data_pedido) = '2026-07'
GROUP BY strftime('%Y-%m', p.data_pedido), p.canal
ON CONFLICT (mes, canal) DO UPDATE SET
    pedidos     = excluded.pedidos,
    faturamento = excluded.faturamento
""")

sql("SELECT * FROM resumo_mensal WHERE mes = '2026-07' ORDER BY canal")

In [ ]:
# DO NOTHING: ignora o conflito silenciosamente (útil em carga incremental)
sql("""
INSERT INTO categorias (nome, margem_alvo) VALUES ('Notebooks', 0.99)
ON CONFLICT (nome) DO NOTHING
""")
sql("SELECT * FROM categorias WHERE nome = 'Notebooks'")

## 6. Índices

Um índice é uma **estrutura auxiliar ordenada** que permite ao banco encontrar linhas sem varrer a tabela inteira.

```
SEM ÍNDICE (full table scan)          COM ÍNDICE (busca em árvore B)
─────────────────────────────         ──────────────────────────────
linha 1    ← olha                              [50]
linha 2    ← olha                             /    \
linha 3    ← olha                         [25]      [75]
...                                       /  \      /  \
linha 999.999 ← olha                    ...  ...  ...  [87] ← achou
linha 1.000.000 ← ACHOU

   1.000.000 leituras                      ~20 leituras
```

Em uma tabela de 1 milhão de linhas, é a diferença entre 2 segundos e 2 milissegundos.

In [ ]:
sql("""
CREATE INDEX idx_pedidos_cliente ON pedidos(cliente_id);
""")
sql("CREATE INDEX idx_pedidos_data   ON pedidos(data_pedido)")
sql("CREATE INDEX idx_pedidos_status ON pedidos(status)")
sql("CREATE INDEX idx_itens_pedido   ON itens_pedido(pedido_id)")
sql("CREATE INDEX idx_itens_produto  ON itens_pedido(produto_id)")

sql("SELECT name, tbl_name FROM sqlite_master WHERE type = 'index' AND name LIKE 'idx_%'")

### O custo: índice não é grátis

| Ganho | Custo |
|-------|-------|
| `SELECT` muito mais rápido | Ocupa espaço em disco |
| `JOIN` mais rápido | **`INSERT`, `UPDATE` e `DELETE` ficam mais lentos** — cada índice precisa ser atualizado |
| `ORDER BY` sem ordenação extra | Manutenção (o banco precisa reequilibrar a árvore) |

> ⚖️ **A regra:** indexe o que você **consulta com frequência**, não tudo. Uma tabela com 12 índices é uma tabela cujos inserts arrastam.

### O que vale indexar

| Indexe | Não indexe |
|--------|-----------|
| Chaves estrangeiras (quase sempre) | Colunas com poucos valores distintos (ex.: `ativo` 0/1) |
| Colunas usadas no `WHERE` com frequência | Tabelas muito pequenas (o scan já é rápido) |
| Colunas de `ORDER BY` recorrente | Colunas raramente consultadas |
| Colunas de `JOIN` | Tabelas com muito mais escrita que leitura |

> 📌 **PK já é indexada automaticamente.** `UNIQUE` também cria um índice. Mas **FK não** — na maioria dos bancos, é você quem precisa criar. Esse é o índice esquecido mais comum, e a causa mais frequente de JOIN lento.

In [ ]:
# Índice COMPOSTO: a ordem das colunas importa
sql("CREATE INDEX idx_pedidos_status_data ON pedidos(status, data_pedido)")

> 💡 **Regra do "prefixo mais à esquerda".** Um índice em `(status, data_pedido)` serve para:
>
> - `WHERE status = 'pago'` ✅
> - `WHERE status = 'pago' AND data_pedido > '2026-07-01'` ✅
> - `WHERE data_pedido > '2026-07-01'` ❌ (não usa o índice)
>
> É como uma lista telefônica ordenada por (sobrenome, nome): serve para achar "Silva", serve para "Silva, João", mas não ajuda em nada para achar todos os "João".
>
> **Coloque primeiro a coluna usada com igualdade, depois a de intervalo.**

In [ ]:
# Índice PARCIAL: indexa só um subconjunto. Menor e mais rápido.
sql("""
CREATE INDEX idx_pedidos_pagos
ON pedidos(data_pedido)
WHERE status = 'pago'
""")

# Índice ÚNICO: garante unicidade E acelera
sql("CREATE UNIQUE INDEX idx_produtos_sku ON produtos(sku)")
sql("SELECT COUNT(*) AS indices FROM sqlite_master WHERE type='index' AND name LIKE 'idx_%'")

## 7. `EXPLAIN QUERY PLAN` — como o banco vai executar

Antes de otimizar, **meça**. O `EXPLAIN QUERY PLAN` mostra a estratégia escolhida pelo otimizador.

O que procurar:

| Aparece | Significa |
|---------|-----------|
| `SCAN tabela` | 🔴 Varredura completa — leu tudo |
| `SEARCH tabela USING INDEX ...` | ✅ Usou índice |
| `USING COVERING INDEX` | ✅✅ Nem precisou tocar na tabela |
| `USE TEMP B-TREE FOR ORDER BY` | 🟡 Ordenou em memória (pode ser evitável) |

In [ ]:
print("① Consulta por coluna INDEXADA:")
sql("EXPLAIN QUERY PLAN SELECT * FROM pedidos WHERE cliente_id = 5")

print("\n② Consulta por coluna SEM índice:")
sql("EXPLAIN QUERY PLAN SELECT * FROM clientes WHERE cidade = 'Campinas'")

In [ ]:
sql("CREATE INDEX idx_clientes_cidade ON clientes(cidade)")
print("Depois de criar o índice:")
sql("EXPLAIN QUERY PLAN SELECT * FROM clientes WHERE cidade = 'Campinas'")

In [ ]:
# Um JOIN: veja como o otimizador escolheu a ordem das tabelas
sql("""
EXPLAIN QUERY PLAN
SELECT c.nome, COUNT(*) FROM pedidos p
JOIN clientes c ON c.id = p.cliente_id
WHERE p.status = 'pago'
GROUP BY c.id, c.nome
""")

### 🔴 Quando o índice **não** é usado

Três situações que anulam o índice — e que aparecem o tempo todo em código real:

In [ ]:
print("① Função aplicada sobre a coluna — o índice não serve:")
sql("EXPLAIN QUERY PLAN SELECT * FROM clientes WHERE upper(cidade) = 'CAMPINAS'")

print("\n② LIKE começando com % — não dá para usar ordenação:")
sql("EXPLAIN QUERY PLAN SELECT * FROM clientes WHERE cidade LIKE '%pinas'")

print("\n③ LIKE com prefixo fixo — este consegue usar:")
sql("EXPLAIN QUERY PLAN SELECT * FROM clientes WHERE cidade LIKE 'Camp%'")

> 🧭 **Por que a função anula o índice?** O índice guarda os valores **como estão** na coluna. Se você pergunta por `upper(cidade)`, o banco precisaria aplicar `upper()` em cada valor para comparar — ou seja, ler todos.
>
> **Soluções:**
> - Normalize na **gravação** (guarde já em minúsculas) em vez de na leitura.
> - Ou crie um **índice de expressão**: `CREATE INDEX idx ON clientes(upper(cidade))`.
>
> O mesmo vale para `WHERE preco * 1.1 > 100` — reescreva como `WHERE preco > 100 / 1.1`, deixando a coluna sozinha de um lado.

In [ ]:
# Índice de expressão resolve o caso do upper()
sql("CREATE INDEX idx_clientes_cidade_upper ON clientes(upper(cidade))")
sql("EXPLAIN QUERY PLAN SELECT * FROM clientes WHERE upper(cidade) = 'CAMPINAS'")

## 8. Transações e ACID

Uma **transação** é um conjunto de operações que acontecem **todas ou nenhuma**.

O exemplo canônico: registrar um pedido exige inserir em `pedidos`, inserir N linhas em `itens_pedido` e dar baixa no estoque. Se a energia cair no meio, você **não pode** ficar com o pedido gravado sem os itens.

### ACID

| Letra | Propriedade | Significa |
|-------|-------------|-----------|
| **A** | Atomicidade | Tudo ou nada. Falhou no meio? Desfaz tudo. |
| **C** | Consistência | O banco vai de um estado válido a outro. Constraints são respeitadas. |
| **I** | Isolamento | Transações simultâneas não veem o estado intermediário uma da outra. |
| **D** | Durabilidade | Depois do `COMMIT`, está gravado — mesmo que a máquina desligue. |

```sql
BEGIN;
    -- várias operações
COMMIT;     -- confirma tudo
-- ou
ROLLBACK;   -- desfaz tudo
```

In [ ]:
import sqlite3

# ⚠️ O módulo sqlite3 do Python gerencia transações automaticamente,
#    o que atrapalha a demonstração. isolation_level=None desliga isso
#    e nos dá controle manual explícito.
CONN.isolation_level = None

sql("SELECT COUNT(*) AS pedidos_antes FROM pedidos")

CONN.execute("BEGIN")
CONN.execute("""INSERT INTO pedidos (cliente_id, data_pedido, status, canal, frete)
                VALUES (1, '2026-08-12', 'pago', 'site', 9.90)""")
CONN.execute("""INSERT INTO pedidos (cliente_id, data_pedido, status, canal, frete)
                VALUES (2, '2026-08-12', 'pago', 'app', 0)""")

print("Dentro da transação (ainda não confirmada):")
sql("SELECT COUNT(*) AS pedidos_durante FROM pedidos")

CONN.execute("ROLLBACK")

print("\nDepois do ROLLBACK:")
sql("SELECT COUNT(*) AS pedidos_depois FROM pedidos")

In [ ]:
# Agora com COMMIT
CONN.execute("BEGIN")
CONN.execute("""INSERT INTO pedidos (cliente_id, data_pedido, status, canal, frete)
                VALUES (1, '2026-08-12', 'pago', 'site', 9.90)""")
novo_id = CONN.execute("SELECT last_insert_rowid()").fetchone()[0]
CONN.execute("""INSERT INTO itens_pedido (pedido_id, produto_id, quantidade, preco_unitario)
                VALUES (?, 1, 2, 2599.90)""", (novo_id,))
CONN.execute("UPDATE produtos SET estoque = estoque - 2 WHERE id = 1")
CONN.execute("COMMIT")

print(f"Pedido {novo_id} gravado com sucesso:")
sql("""
SELECT p.id, p.data_pedido, pr.nome, i.quantidade, pr.estoque AS estoque_atual
FROM pedidos p
JOIN itens_pedido i ON i.pedido_id = p.id
JOIN produtos pr    ON pr.id = i.produto_id
WHERE p.id = ?
""", (novo_id,))

### O padrão correto em Python: `try/except/rollback`

Uma transação que falha no meio **precisa** de `ROLLBACK`. Se você só capturar a exceção e seguir, a transação fica aberta e trava a tabela.

In [ ]:
def registrar_pedido(cliente_id, canal, itens):
    """Registra um pedido com seus itens, atomicamente.

    itens: lista de (produto_id, quantidade)
    Retorna o id do pedido, ou levanta a exceção original após desfazer tudo.
    """
    try:
        CONN.execute("BEGIN")

        CONN.execute(
            "INSERT INTO pedidos (cliente_id, data_pedido, status, canal, frete) "
            "VALUES (?, date('now'), 'pendente', ?, 0)",
            (cliente_id, canal),
        )
        pedido_id = CONN.execute("SELECT last_insert_rowid()").fetchone()[0]

        for produto_id, quantidade in itens:
            preco = CONN.execute(
                "SELECT preco FROM produtos WHERE id = ?", (produto_id,)
            ).fetchone()[0]

            CONN.execute(
                "INSERT INTO itens_pedido (pedido_id, produto_id, quantidade, preco_unitario) "
                "VALUES (?, ?, ?, ?)",
                (pedido_id, produto_id, quantidade, preco),
            )
            # O CHECK (estoque >= 0) vai barrar se não houver saldo
            CONN.execute(
                "UPDATE produtos SET estoque = estoque - ? WHERE id = ?",
                (quantidade, produto_id),
            )

        CONN.execute("COMMIT")
        return pedido_id

    except Exception:
        CONN.execute("ROLLBACK")     # ← sem isto, a transação fica aberta
        raise


# ① Pedido válido
pedido = registrar_pedido(3, "site", [(5, 2), (9, 3)])
print(f"✅ Pedido {pedido} registrado")
sql("SELECT id, sku, nome, estoque FROM produtos WHERE id IN (5, 9)")

In [ ]:
# ② Pedido que estoura o estoque no MEIO da operação
print("Estoque antes:")
sql("SELECT id, sku, estoque FROM produtos WHERE id IN (5, 4)")

try:
    registrar_pedido(3, "site", [(5, 1), (4, 99999)])   # o produto 4 não tem 99999
except sqlite3.IntegrityError as erro:
    print(f"\n❌ Falhou: {erro}")

print("\nEstoque depois — repare que o produto 5 NÃO foi debitado:")
sql("SELECT id, sku, estoque FROM produtos WHERE id IN (5, 4)")

> 💡 **É isso que a atomicidade compra.** O primeiro item já tinha sido inserido e o estoque já tinha sido debitado quando o segundo falhou. O `ROLLBACK` desfez tudo. Sem transação, você ficaria com um pedido pela metade e estoque errado — e descobriria semanas depois, na conferência.

### `SAVEPOINT` — transações aninhadas

Um `SAVEPOINT` é um marco dentro da transação. Você pode voltar até ele sem abortar tudo.

In [ ]:
CONN.execute("BEGIN")
CONN.execute("INSERT INTO categorias (nome, margem_alvo) VALUES ('Temporária A', 0.10)")

CONN.execute("SAVEPOINT antes_da_b")
CONN.execute("INSERT INTO categorias (nome, margem_alvo) VALUES ('Temporária B', 0.20)")

print("Com A e B:")
sql("SELECT nome FROM categorias WHERE nome LIKE 'Temporária%'")

CONN.execute("ROLLBACK TO antes_da_b")     # desfaz só o B
print("\nDepois do ROLLBACK TO (só B foi desfeito):")
sql("SELECT nome FROM categorias WHERE nome LIKE 'Temporária%'")

CONN.execute("ROLLBACK")                   # agora desfaz tudo
print("\nDepois do ROLLBACK completo:")
sql("SELECT nome FROM categorias WHERE nome LIKE 'Temporária%'")

### Isolamento e concorrência

O **I** de ACID trata do que uma transação enxerga enquanto outra está rodando.

| Fenômeno | O que é |
|----------|---------|
| *Dirty read* | Ler dado não confirmado de outra transação |
| *Non-repeatable read* | Ler a mesma linha duas vezes e obter valores diferentes |
| *Phantom read* | A mesma consulta devolver linhas novas na segunda execução |

Bancos oferecem **níveis de isolamento** (`READ UNCOMMITTED`, `READ COMMITTED`, `REPEATABLE READ`, `SERIALIZABLE`) que trocam consistência por desempenho.

> ℹ️ **No SQLite isso é simples porque a concorrência é limitada:** ele permite **vários leitores simultâneos, mas apenas um escritor**. O nível efetivo é `SERIALIZABLE` — o mais rigoroso.
>
> Isso é ótimo para correção e é exatamente o motivo pelo qual o SQLite não serve para uma API com muitos usuários escrevendo ao mesmo tempo. No **Módulo 05** você vai ver como o PostgreSQL resolve isso com MVCC.
>
> 💡 O modo **WAL** (`PRAGMA journal_mode = WAL`) melhora bastante a situação: leitores não bloqueiam o escritor e vice-versa. É a primeira configuração que se liga em qualquer SQLite de produção.

In [ ]:
sql("PRAGMA journal_mode")

## 9. `sqlite3` no Python — cuidados de produção

In [ ]:
%%writefile repositorio_demo.py
"""Camada de acesso a dados — o padrão que o Atlas vai usar no M03.

Três princípios:
  1. SQL fica em UM lugar (esta camada), não espalhado pelo sistema.
  2. TODA consulta é parametrizada.
  3. Conexões e transações são gerenciadas por context managers.
"""

import sqlite3
from contextlib import contextmanager
from pathlib import Path


@contextmanager
def conectar(caminho: str | Path):
    """Abre uma conexão configurada e garante o fechamento."""
    conexao = sqlite3.connect(caminho)
    try:
        # Sempre, em toda conexão:
        conexao.execute("PRAGMA foreign_keys = ON")
        # WAL: leitores não bloqueiam o escritor
        conexao.execute("PRAGMA journal_mode = WAL")
        # row_factory: acesso por NOME da coluna em vez de índice
        conexao.row_factory = sqlite3.Row
        yield conexao
    finally:
        conexao.close()


@contextmanager
def transacao(conexao: sqlite3.Connection):
    """Commit no sucesso, rollback em qualquer exceção."""
    try:
        yield conexao
        conexao.commit()
    except Exception:
        conexao.rollback()
        raise


def buscar_produtos_por_categoria(conexao, categoria_id: int) -> list[dict]:
    """Consulta parametrizada devolvendo dicionários."""
    cursor = conexao.execute(
        "SELECT id, sku, nome, preco, estoque "
        "FROM produtos WHERE categoria_id = ? AND ativo = 1 "
        "ORDER BY nome",
        (categoria_id,),
    )
    return [dict(linha) for linha in cursor.fetchall()]


def inserir_produtos_em_lote(conexao, produtos: list[tuple]) -> int:
    """executemany: MUITO mais rápido que N execute() separados."""
    with transacao(conexao):
        cursor = conexao.executemany(
            "INSERT INTO produtos (sku, nome, categoria_id, preco, custo, estoque) "
            "VALUES (?, ?, ?, ?, ?, ?)",
            produtos,
        )
        return cursor.rowcount


if __name__ == "__main__":
    print("Módulo de exemplo — importe as funções, não execute diretamente.")

In [ ]:
# row_factory: acesso por nome em vez de índice numérico
CONN.row_factory = sqlite3.Row

cursor = CONN.execute("SELECT id, sku, nome, preco FROM produtos LIMIT 3")
for linha in cursor:
    print(f"{linha['sku']:<14} {linha['nome']:<30} R$ {linha['preco']:>8,.2f}")

CONN.row_factory = None   # volta ao padrão para o resto do notebook

In [ ]:
# executemany + medição: o ganho de inserir em lote
import time

ddl("CREATE TABLE bench (id INTEGER PRIMARY KEY, valor TEXT)")

dados = [(f"linha {i}",) for i in range(5000)]

# ① Um insert por vez, cada um em sua própria transação (autocommit).
#    É o que acontece quando você não abre transação explícita.
inicio = time.perf_counter()
for d in dados:
    CONN.execute("INSERT INTO bench (valor) VALUES (?)", d)
tempo_loop = time.perf_counter() - inicio

CONN.execute("DELETE FROM bench")

# ② Tudo em UMA transação, com executemany
inicio = time.perf_counter()
CONN.execute("BEGIN")
CONN.executemany("INSERT INTO bench (valor) VALUES (?)", dados)
CONN.execute("COMMIT")
tempo_batch = time.perf_counter() - inicio

print(f"5.000 inserts individuais : {tempo_loop*1000:8.1f} ms")
print(f"executemany em transação  : {tempo_batch*1000:8.1f} ms")
print(f"Ganho                     : {tempo_loop/max(tempo_batch, 1e-9):8.1f}x")

ddl("DROP TABLE bench")

> 💡 **Por que a diferença?** Sem transação explícita, o SQLite abre e confirma uma transação **por comando** — e cada `COMMIT` força uma sincronização com o meio de armazenamento.
>
> ⚠️ **O número que você viu está MUITO subestimado.** Nosso banco está em `:memory:`, onde não há disco envolvido — por isso o ganho fica na casa de 3–5×. **Em um banco em arquivo, a mesma comparação costuma dar 50× a 200×**, porque cada commit vira um `fsync()` de verdade.
>
> Teste você mesmo: troque `sqlite3.connect(":memory:")` por `sqlite3.connect("teste.db")` e rode de novo.
>
> Essa é a otimização número 1 de qualquer carga de dados. Guarde para o Módulo 10.

## 🔧 Prática guiada — Migrando o Atlas do CSV para o banco

Este é exatamente o trabalho do Módulo 03 no projeto: pegar os CSVs e carregá-los em um schema relacional, de forma **idempotente** (pode rodar duas vezes sem duplicar).

In [ ]:
%%writefile carga_atlas.py
"""Carga do CSV de vendas da Aurora para o banco relacional.

Demonstra o padrão de ETL que você vai reencontrar no Módulo 10:
    extrair -> transformar -> carregar, tudo em UMA transação.
"""

import csv
import sqlite3
from pathlib import Path

SCHEMA = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS clientes (
    id     INTEGER PRIMARY KEY,
    nome   TEXT NOT NULL,
    email  TEXT NOT NULL UNIQUE,
    cidade TEXT NOT NULL,
    uf     TEXT NOT NULL CHECK (length(uf) = 2)
);

CREATE TABLE IF NOT EXISTS produtos (
    id    INTEGER PRIMARY KEY,
    nome  TEXT NOT NULL UNIQUE,
    preco REAL NOT NULL CHECK (preco >= 0)
);

CREATE TABLE IF NOT EXISTS pedidos (
    id          INTEGER PRIMARY KEY,
    cliente_id  INTEGER NOT NULL REFERENCES clientes(id),
    produto_id  INTEGER NOT NULL REFERENCES produtos(id),
    quantidade  INTEGER NOT NULL CHECK (quantidade > 0),
    preco_unit  REAL    NOT NULL CHECK (preco_unit >= 0),
    status      TEXT    NOT NULL CHECK (status IN ('pago','pendente','cancelado')),
    data_pedido TEXT    NOT NULL
);

CREATE INDEX IF NOT EXISTS idx_pedidos_cliente ON pedidos(cliente_id);
CREATE INDEX IF NOT EXISTS idx_pedidos_status  ON pedidos(status);
"""


def carregar(caminho_csv: Path, caminho_db: Path) -> dict:
    conexao = sqlite3.connect(caminho_db)
    conexao.execute("PRAGMA foreign_keys = ON")
    conexao.executescript(SCHEMA)

    validos, rejeitados = 0, []

    try:
        conexao.execute("BEGIN")

        with open(caminho_csv, newline="", encoding="utf-8") as arquivo:
            for numero, linha in enumerate(csv.DictReader(arquivo), start=2):
                try:
                    # ── cliente: insere se não existe, devolve o id ──
                    conexao.execute(
                        "INSERT INTO clientes (nome, email, cidade, uf) VALUES (?,?,?,?) "
                        "ON CONFLICT (email) DO NOTHING",
                        (linha["cliente"].strip().title(),
                         linha["email"].strip().lower(),
                         linha["cidade"].strip().title(),
                         linha["uf"].strip().upper()),
                    )
                    cliente_id = conexao.execute(
                        "SELECT id FROM clientes WHERE email = ?",
                        (linha["email"].strip().lower(),),
                    ).fetchone()[0]

                    # ── produto ──
                    conexao.execute(
                        "INSERT INTO produtos (nome, preco) VALUES (?,?) "
                        "ON CONFLICT (nome) DO UPDATE SET preco = excluded.preco",
                        (linha["produto"].strip(), float(linha["preco"])),
                    )
                    produto_id = conexao.execute(
                        "SELECT id FROM produtos WHERE nome = ?",
                        (linha["produto"].strip(),),
                    ).fetchone()[0]

                    # ── pedido (idempotente pelo id do CSV) ──
                    conexao.execute(
                        "INSERT INTO pedidos (id, cliente_id, produto_id, quantidade, "
                        "preco_unit, status, data_pedido) VALUES (?,?,?,?,?,?,?) "
                        "ON CONFLICT (id) DO UPDATE SET "
                        "  quantidade = excluded.quantidade, "
                        "  preco_unit = excluded.preco_unit, "
                        "  status     = excluded.status",
                        (int(linha["id"]), cliente_id, produto_id,
                         int(linha["quantidade"]), float(linha["preco"]),
                         linha["status"].strip().lower(), linha["data"].strip()),
                    )
                    validos += 1

                except (ValueError, KeyError, sqlite3.IntegrityError) as erro:
                    rejeitados.append({"linha": numero, "motivo": f"{type(erro).__name__}: {erro}"})

        conexao.execute("COMMIT")

    except Exception:
        conexao.execute("ROLLBACK")
        raise
    finally:
        conexao.close()

    return {"validos": validos, "rejeitados": rejeitados}

In [ ]:
# Criamos um CSV de exemplo e rodamos a carga DUAS VEZES
import csv as _csv
from pathlib import Path as _Path
import carga_atlas

_Path("dados_m03").mkdir(exist_ok=True)
_csv_path = _Path("dados_m03/vendas.csv")
_db_path = _Path("dados_m03/atlas.db")
_db_path.unlink(missing_ok=True)

with open(_csv_path, "w", newline="", encoding="utf-8") as f:
    escritor = _csv.writer(f)
    escritor.writerow(["id", "data", "cliente", "email", "cidade", "uf",
                       "produto", "quantidade", "preco", "status"])
    escritor.writerows([
        [1001, "2026-07-01", "maria souza", "MARIA@X.COM", "campinas", "sp", "Notebook Dell", 2, 2599.90, "PAGO"],
        [1002, "2026-07-01", "joão lima", "joao@x.com", "são paulo", "SP", "Mouse Logitech", 10, 89.90, "pago"],
        [1003, "2026-07-02", "maria souza", "maria@x.com", "Campinas", "SP", "Teclado", 3, 249.00, "cancelado"],
        [1004, "2026-07-03", "ana costa", "ana@x.com", "Sorocaba", "SP", "Monitor LG", 1, 1199.00, "pago"],
        [1005, "2026-07-03", "", "ruim@x.com", "Santos", "SP", "Webcam", -2, 449.00, "pago"],
    ])

print("── Primeira execução ──")
r1 = carga_atlas.carregar(_csv_path, _db_path)
print(f"válidos: {r1['validos']} | rejeitados: {len(r1['rejeitados'])}")
for e in r1["rejeitados"]:
    print("  ", e["linha"], e["motivo"])

print("\n── Segunda execução (idempotência) ──")
r2 = carga_atlas.carregar(_csv_path, _db_path)
print(f"válidos: {r2['validos']} | rejeitados: {len(r2['rejeitados'])}")

In [ ]:
# Conferindo: rodar duas vezes NÃO duplicou nada
import sqlite3 as _sq

_con = _sq.connect(_db_path)
for _t in ["clientes", "produtos", "pedidos"]:
    print(f"{_t:<10}", _con.execute(f"SELECT COUNT(*) FROM {_t}").fetchone()[0])

print("\nRelatório a partir do banco carregado:")
for _linha in _con.execute("""
    SELECT c.cidade,
           COUNT(*)                              AS pedidos,
           ROUND(SUM(p.quantidade * p.preco_unit), 2) AS faturamento
    FROM pedidos p
    JOIN clientes c ON c.id = p.cliente_id
    WHERE p.status = 'pago'
    GROUP BY c.cidade
    ORDER BY faturamento DESC
"""):
    print(f"  {_linha[0]:<12} {_linha[1]:>3} pedidos   R$ {_linha[2]:>10,.2f}")
_con.close()

> 💡 **Repare no que a carga faz:** normaliza texto (`.title()`, `.lower()`), usa `ON CONFLICT` para ser idempotente, valida com constraints do banco, rejeita a linha ruim sem derrubar o processo, e envolve tudo em **uma transação**. É o mesmo desenho do pipeline que você vai construir no Módulo 10 — só que lá com muito mais volume.

## 📝 Exercícios

**E1.** Insira 3 produtos novos em uma categoria à sua escolha, usando um único `INSERT` com múltiplos `VALUES`.

**E2.** Escreva uma consulta parametrizada que receba `uf` e `preco_minimo` como parâmetros nomeados e liste os produtos comprados por clientes daquela UF acima daquele preço.

**E3.** Demonstre a SQL injection: monte uma consulta por concatenação e faça uma entrada maliciosa retornar linhas que não deveria. Depois corrija com placeholder.

**E4.** Aplique um reajuste de 5% em todos os produtos da categoria "Periféricos" que tenham estoque acima de 50. **Antes**, escreva o `SELECT` equivalente e confirme quantas linhas serão afetadas.

**E5.** Implemente soft delete: marque como inativo todo produto sem nenhuma venda. Depois crie uma view que mostre só os ativos.

**E6.** Use `UPSERT` para atualizar a tabela `resumo_mensal` referente ao mês 2026-06. Rode duas vezes e prove que não duplicou.

**E7.** Crie um índice composto em `itens_pedido(pedido_id, produto_id)` e use `EXPLAIN QUERY PLAN` para mostrar a diferença em uma consulta que filtra pelos dois.

**E8.** Escreva uma função Python `transferir_estoque(origem_id, destino_id, quantidade)` que debite de um produto e credite em outro, **dentro de uma transação**. Teste o caso de falha (estoque insuficiente) e prove que nada foi alterado.

**E9.** Meça o ganho de `executemany` + transação contra inserts individuais, com 10.000 linhas.

**E10.** Encontre uma consulta lenta no banco (use `EXPLAIN QUERY PLAN` para achar um `SCAN`), crie o índice adequado e mostre o plano antes e depois.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

## 📋 Cola de referência

```sql
-- ── INSERT ──
INSERT INTO t (a, b) VALUES (1, 'x');
INSERT INTO t (a, b) VALUES (1,'x'), (2,'y'), (3,'z');   -- múltiplo
INSERT INTO t (a, b) SELECT c, d FROM outra WHERE ...;   -- de consulta
INSERT INTO t (a) VALUES (1) RETURNING id;               -- SQLite 3.35+

-- ── UPSERT ──
INSERT INTO t (chave, valor) VALUES (?, ?)
ON CONFLICT (chave) DO UPDATE SET valor = excluded.valor;
ON CONFLICT (chave) DO NOTHING;

-- ── UPDATE ──
UPDATE t SET a = 1, b = b * 1.1 WHERE id = 5;
-- ⚠️ SEMPRE escreva o SELECT com o mesmo WHERE antes

-- ── DELETE ──
DELETE FROM t WHERE id = 5;
DELETE FROM t;             -- ⚠️ esvazia a tabela
DROP TABLE t;              -- ⚠️ remove a tabela

-- ── Soft delete ──
UPDATE t SET deletado_em = date('now') WHERE id = 5;
CREATE VIEW t_ativos AS SELECT * FROM t WHERE deletado_em IS NULL;

-- ── Índices ──
CREATE INDEX idx_nome ON t(coluna);
CREATE INDEX idx_comp ON t(a, b);            -- prefixo mais à esquerda
CREATE UNIQUE INDEX idx_u ON t(coluna);
CREATE INDEX idx_parcial ON t(a) WHERE b = 'x';
CREATE INDEX idx_expr ON t(upper(coluna));
DROP INDEX idx_nome;

EXPLAIN QUERY PLAN SELECT ...;    -- SCAN = ruim, SEARCH USING INDEX = bom

-- ── Transações ──
BEGIN;
    ...
COMMIT;      -- ou ROLLBACK;

SAVEPOINT marco;
ROLLBACK TO marco;
RELEASE marco;

-- ── PRAGMAs de produção ──
PRAGMA foreign_keys = ON;      -- SEMPRE
PRAGMA journal_mode = WAL;     -- leitores não bloqueiam escritor
```

```python
# ── Python ──
conn = sqlite3.connect("atlas.db")
conn.execute("PRAGMA foreign_keys = ON")
conn.row_factory = sqlite3.Row          # acesso por nome de coluna

# ✅ SEMPRE parametrizado
conn.execute("SELECT * FROM t WHERE id = ?", (5,))
conn.execute("SELECT * FROM t WHERE id = :id", {"id": 5})
conn.executemany("INSERT INTO t VALUES (?,?)", lista_de_tuplas)

# ❌ NUNCA concatene
conn.execute(f"SELECT * FROM t WHERE id = {id}")

# Transação com segurança
try:
    conn.execute("BEGIN")
    ...
    conn.execute("COMMIT")
except Exception:
    conn.execute("ROLLBACK")
    raise
```

## ✅ Checklist de saída

- [ ] Sempre listo as colunas no `INSERT`
- [ ] Uso `INSERT ... SELECT` para popular a partir de consultas
- [ ] 🔴 **Uso placeholders em 100% das consultas com dados externos**
- [ ] Sei explicar o que é SQL injection e por que placeholder resolve
- [ ] Sei que nome de tabela/coluna não pode ser parametrizado, e uso lista branca
- [ ] **Escrevo o `SELECT` antes de todo `UPDATE`/`DELETE`**
- [ ] Entendo quando soft delete é preferível, e o custo dele
- [ ] Uso `ON CONFLICT` para cargas idempotentes
- [ ] Sei o que um índice ganha e o que ele custa
- [ ] Sei que FK **não** ganha índice automático
- [ ] Entendo a regra do prefixo mais à esquerda em índice composto
- [ ] Leio `EXPLAIN QUERY PLAN` e reconheço `SCAN` vs `SEARCH USING INDEX`
- [ ] Sei três situações em que o índice não é usado
- [ ] Explico ACID e sei quando abrir uma transação
- [ ] Escrevo `try / commit / except / rollback / raise`
- [ ] Uso `executemany` dentro de transação para cargas em lote

---

### ➡️ Próxima etapa

**`03_99_Lista_Exercicios.ipynb`** — modelagem ER, consultas completas e o projeto do módulo: migrar o Atlas de CSV para SQLite.